# Semifinal 05 - Pairwise cosine similarity between tweet embeddings to build the lexical-mutation graph

## Import Libraries

In [ ]:
DATA_DIR = "../../data/semifinal"

In [2]:
%pip install -q datasets
# %pip install --upgrade -q pyarrow

In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

# import tensorflow as tf
# from catboost import CatBoostClassifier, Pool
import torch
import torch.nn.functional as F

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    recall_score,
    precision_score,
    balanced_accuracy_score,
    confusion_matrix
)

import datasets
from transformers import (
    AutoFeatureExtractor,
    AutoModel,
    AutoImageProcessor,
    AutoProcessor,
    AutoTokenizer,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


## Helpers

In [ ]:
def average_pool(last_hidden_states: torch.Tensor,
                 attention_mask: torch.Tensor) -> torch.Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]


def extract_text_embeddings(
        model,
        feature_extractor,
        text_name="description",
):
    device = model.device

    def pp(batch):
        inputs = feature_extractor(
            text=batch[text_name],
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        # embeddings = model(**inputs).last_hidden_state[:, 0, :].cpu()
        outputs = model(**inputs)
        embeddings = average_pool(outputs.last_hidden_state, inputs.attention_mask)
        embeddings = F.normalize(embeddings, p=2, dim=1)
        return {"embedding": embeddings}

    return pp


def huggingface_text_embedding(
        df,
        text_name="description",
        modelname="intfloat/multilingual-e5-base",
        batched=True,
        batch_size=24,
):
    # initialize huggingface model
    feature_extractor = AutoTokenizer.from_pretrained(modelname)
    model = AutoModel.from_pretrained(modelname, output_hidden_states=True)
    # create huggingface dataset from df
    dataset = datasets.Dataset.from_pandas(df)
    # compute embedding
    device = "cuda" if torch.cuda.is_available() else "cpu"
    extract_fn = extract_text_embeddings(model.to(device), feature_extractor, text_name)
    updated_dataset = dataset.map(extract_fn, batched=batched, batch_size=batch_size)
    df_temp = updated_dataset.to_pandas()
    df_emb = pd.DataFrame()
    df_emb["embedding"] = df_temp["embedding"]
    return df_emb

## Load Dataset

In [ ]:
df = pd.read_csv(f"{DATA_DIR}/encrypted_merge_no_rt.csv")
df.head()

   Unnamed: 0  index            created_at  \
0           0     21  2024-01-04T09:57:18Z   
1           1     34  2024-01-04T09:57:22Z   
2           2     38  2024-01-04T09:57:23Z   
3           3     39  2024-01-04T09:57:23Z   
4           4     40  2024-01-04T09:57:24Z   

                                        username    tcode  num_retweets  type  \
0  @YMN1SQke21evlTScXsu3cVvR/l6EPOAYa7EQLb0d5ss=  mention             0  twit   
1  @7WZM6I5Q/iYrH1SMgclp34S65jhXMMyhXdvYbR3TO9s=    reply             0  twit   
2  @5bcvctYrjL1R7+juB8TjhVbDqjgCOCKLMJVerK5CqF4=    reply             0  twit   
3  @frAtGFoe5QExA7+rfzOX6vqrbItjXQdZkORwO0DWHW0=    reply             0  twit   
4  @gxme/2Y9YADfRdCNIhbkqYScL8zFOVZ78mJuoTanlps=  mention             0  twit   

   frn_cnt  flw_cnt  sts_cnt                     loc  lst_cnt  \
0    128.0      0.0    819.0  Surakarta, Jawa Tengah      0.0   
1    347.0    207.0   6378.0                semarang      1.0   
2    153.0    124.0    319.0             

In [ ]:
with open(f"{DATA_DIR}/index.pkl", "wb") as f:
    pickle.dump(df["index"].to_list(), f)

In [ ]:
del df

In [ ]:
index_lst = pickle.load(open(f"{DATA_DIR}/index.pkl", "rb"))
len(index_lst)

3319644

In [ ]:
with open(f"{DATA_DIR}/embeddings/embeddings.pkl", "rb") as f:
    embeddings = pickle.load(f)

## Testing

In [ ]:
df.loc[0, "content_cleaned_with_query"]

'query: Mahfud MD dan Ganjar Pranowo menunjukkan bahwa keceriaan adalah kunci untuk mendekatkan diri pada rakyat Ganjar Mahfud Hebat Buruh Naik Kelas'

In [ ]:
df.loc[1:2, "content_cleaned_with_query"]

1    query: Ga capek urus Prabowo Mending urus Ganj...
2    query: Dalam pembangunan jalan nasional Anies ...
Name: content_cleaned_with_query, dtype: object

In [ ]:
cosine_similarity(
    [df.loc[0, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.82390547, 0.9164641 ]], dtype=float32)

In [ ]:
tmp_df = pd.DataFrame({
    "content_cleaned": [
        "query: anies telah membangun jalanan nasional",
        "query: Mahfud MD dan Ganjar Pranowo menunjukkan bahwa keceriaan adalah kunci untuk mendekatkan diri pada rakyat Ganjar Mahfud Hebat Buruh Naik Kelas",
        "query: Dalam membangun jalan nasional Anies berencana bekerja sama dengan para ahli lingkungan untuk merancang infrastruktur yang lebih ramah lingkungan.",
        "query: Anies membangun jalan nasional berencana bekerja sama dengan para ahli.",
        "query: In constructing national roads, Anies plans to collaborate with environmental experts to design more environmentally friendly infrastructure."
    ]
})
tmp_df

,content_cleaned
0,query: anies telah membangun jalanan nasional
1,query: Mahfud MD dan Ganjar Pranowo menunjukka...
2,query: Dalam membangun jalan nasional Anies be...
3,query: Anies membangun jalan nasional berencan...
4,"query: In constructing national roads, Anies p..."


In [ ]:
tmp_df_embed = huggingface_text_embedding(
    tmp_df,
    text_name="content_cleaned",
    modelname="intfloat/multilingual-e5-base"
)

tmp_df_embed

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

,embedding
0,"[0.026050605, 0.03216584, -0.019220887, 0.0301..."
1,"[0.017292552, 0.034554314, -0.015211774, 0.044..."
2,"[0.006827455, -0.0088892635, -0.023698606, 0.0..."
3,"[0.0044027315, -0.0037597117, -0.0226008, 0.05..."
4,"[0.03572188, 0.007526107, -0.03143481, 0.02418..."


In [ ]:
cosine_similarity(
    [tmp_df_embed.loc[0, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.80588293, 0.8590038 ]], dtype=float32)

In [ ]:
cosine_similarity(
    [tmp_df_embed.loc[1, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.82390547, 0.9164641 ]], dtype=float32)

In [ ]:
cosine_similarity(
    [tmp_df_embed.loc[2, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.8078041, 0.9912909]], dtype=float32)

In [ ]:
cosine_similarity(
    [tmp_df_embed.loc[3, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.8307289 , 0.96709484]], dtype=float32)

In [ ]:
cosine_similarity(
    [tmp_df_embed.loc[4, "embedding"]],
    df.loc[1:2, "embedding"].to_list()
)

array([[0.74984854, 0.8648253 ]], dtype=float32)

In [ ]:
relation_df = pd.DataFrame()
relation_df["index"] = df["index"]
relation_df.head()

,index
0,21
1,34
2,38
3,39
4,40


In [ ]:
relation_df["tmp"] = [[("1", 0.9)]] * df.shape[0]
relation_df.head()

,index,tmp
0,21,"[(1, 0.9)]"
1,34,"[(1, 0.9)]"
2,38,"[(1, 0.9)]"
3,39,"[(1, 0.9)]"
4,40,"[(1, 0.9)]"


In [ ]:
relation_df.drop(columns=["tmp"], inplace=True)

In [ ]:
del tmp_df_embed, tmp_df

## Compute Cosine Similarity

In [ ]:
for i, row in tqdm(df.iterrows(), total=df.shape[0]):
    # cosine_sims = cosine_similarity(
    #     [row["embedding"]],
    #     df.loc[i + 1:, "embedding"].to_list()
    # ).max()
    cosine_sims = [0.9, 0.8]
    # get pair index and cosine similarity if cosine similarity > 0.85
    pair_index = [
        (df.loc[i + 1 + j, "index"], cosine_sims[j])
        for j in range(cosine_sims.shape[0])
        if cosine_sims[j] > 0.85
    ]
    relation_df.loc[i, "tmp"] = pair_index
    break

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 for i, row in tqdm(df.iterrows(), total=df.shape[0]):                                       │
│    2 │   # cosine_sims = cosine_similarity(                                                      │
│    3 │   #     [row["embedding"]],                                                               │
│    4 │   #     df.loc[i + 1:, "embedding"].to_list()                                             │
│                                                                                                  │
│ d:\miniconda3\envs\torch\lib\site-packages\tqdm\std.py:1178 in __iter__                          │
│                                                                                                  │
│   1175 │   │   time = self._time                                                                 │
│   1176 │   │                                                                                     │
│   1177 │   │   try:                                                                              │
│ ❱ 1178 │   │   │   for obj in iterable:                                                          │
│   1179 │   │   │   │   yield obj                                                                 │
│   1180 │   │   │   │   # Update and possibly print the progressbar.                              │
│   1181 │   │   │   │   # Note: does not call self.update(1) for speed optimisation.              │
│                                                                                                  │
│ d:\miniconda3\envs\torch\lib\site-packages\pandas\core\frame.py:1410 in iterrows                 │
│                                                                                                  │
│    1407 │   │   """                                                                              │
│    1408 │   │   columns = self.columns                                                           │
│    1409 │   │   klass = self._constructor_sliced                                                 │
│ ❱  1410 │   │   for k, v in zip(self.index, self.values):                                        │
│    1411 │   │   │   s = klass(v, index=columns, name=k).__finalize__(self)                       │
│    1412 │   │   │   yield k, s                                                                   │
│    1413                                                                                          │
│                                                                                                  │
│ d:\miniconda3\envs\torch\lib\site-packages\pandas\core\frame.py:11739 in values                  │
│                                                                                                  │
│   11736 │   │   │      ['monkey', nan, None]], dtype=object)                                     │
│   11737 │   │   """                                                                              │
│   11738 │   │   self._consolidate_inplace()                                                      │
│ ❱ 11739 │   │   return self._mgr.as_array()                                                      │
│   11740 │                                                                                        │
│   11741 │   @overload                                                                            │
│   11742 │   def ffill(                                                                           │
│                                                                                                  │
│ d:\miniconda3\envs\torch\lib\site-packages\pandas\core\internals\managers.py:1770 in as_array    │
│                                                            